# HAM Synthetic Experiments Runner

Run this notebook on a GPU to massively accelerate JAX JIT compilation and the multi-iteration inverse problem tests.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure the project src/ is in the python path
project_root = Path(os.getcwd()).parent.parent.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import jax
print("Available JAX devices:", jax.devices())

In [ ]:
from experiments.wildfire.synthetic.run_experiments import ALL_EXPERIMENTS

# ==================================================================
# Configuration
# ==================================================================
categories_to_run = ['A', 'B', 'C', 'D', 'E']  # Select categories, e.g. ['C', 'E']
experiments_to_run = []  # Specific experiment names, e.g. ['C1_isotropic_recovery'] (overrides categories if not empty)

quick_mode = False       # True = smaller grids for rapid prototyping
save_results = True      # Save metrics to summary.json
visualize_results = True # Save PDFs of metric recoveries
verbose = True           # Print solver logs
# ==================================================================

In [ ]:
for name, exp_class in ALL_EXPERIMENTS.items():
    category_letter = exp_class.category[0].upper()
    
    should_run = False
    if len(experiments_to_run) > 0:
        should_run = any(e in name for e in experiments_to_run)
    else:
        should_run = category_letter in categories_to_run
        
    if should_run:
        print(f"\n{'='*60}\nExperiment: {name}\nCategory: {exp_class.category}\n{'='*60}")
        
        # Initialize and run
        exp = exp_class(quick=quick_mode)
        result = exp.execute(save=save_results, visualize=visualize_results, verbose=verbose)
        
        print(f"\nSuccess: {result['success']}")
        print("Metrics:")
        for k, v in result['metrics'].items():
            print(f"  {k}: {v:.6f}")